# Relatório Operacional de Pré-Decolagem

**Atividade Integradora - Fase 1**  
**Grupo 24 - Felipe Cagnin de Lima**

Este notebook simula a leitura da telemetria, procura anomalias e decide se a missão está pronta para decolar. As faixas usadas são premissas acadêmicas desta atividade.

## 1. Faixas seguras

- Temperatura interna: 18 °C a 27 °C
- Temperatura externa: -10 °C a 45 °C
- Integridade estrutural: 1
- Energia mínima: 80%
- Pressão dos tanques: 280 kPa a 320 kPa
- Módulos críticos: todos funcionando

## 2. Funções de verificação e energia

In [1]:
LIMITES = {
    "temperatura_interna": (18, 27),
    "temperatura_externa": (-10, 45),
    "energia_minima": 80,
    "pressao_tanques": (280, 320),
}


def verificar_decolagem(dados):
    anomalias = []

    temp_interna = dados["temperatura_interna"]
    temp_externa = dados["temperatura_externa"]

    if not LIMITES["temperatura_interna"][0] <= temp_interna <= LIMITES["temperatura_interna"][1]:
        anomalias.append(f"Temperatura interna fora da faixa: {temp_interna} °C")

    if not LIMITES["temperatura_externa"][0] <= temp_externa <= LIMITES["temperatura_externa"][1]:
        anomalias.append(f"Temperatura externa fora da faixa: {temp_externa} °C")

    if dados["integridade_estrutural"] != 1:
        anomalias.append("Integridade estrutural comprometida")

    if dados["energia"] < LIMITES["energia_minima"]:
        anomalias.append(f"Energia abaixo do mínimo: {dados['energia']}%")

    minimo, maximo = LIMITES["pressao_tanques"]
    for tanque, pressao in dados["pressao_tanques"].items():
        if not minimo <= pressao <= maximo:
            anomalias.append(f"Pressão do tanque {tanque} fora da faixa: {pressao} kPa")

    for modulo, funcionando in dados["modulos_criticos"].items():
        if not funcionando:
            anomalias.append(f"Módulo crítico com falha: {modulo}")

    status = "PRONTO PARA DECOLAR" if not anomalias else "DECOLAGEM ABORTADA"
    return status, anomalias


def calcular_energia(capacidade_kwh, carga_atual, consumo_decolagem_kwh, perdas_percentuais):
    energia_inicial = capacidade_kwh * carga_atual / 100
    perdas_kwh = energia_inicial * perdas_percentuais / 100
    energia_restante = energia_inicial - perdas_kwh - consumo_decolagem_kwh
    autonomia_percentual = energia_restante / capacidade_kwh * 100
    return energia_inicial, perdas_kwh, energia_restante, autonomia_percentual


def mostrar_resultado(nome, dados):
    status, anomalias = verificar_decolagem(dados)
    print(f"\n=== {nome} ===")
    print(f"Resultado: {status}")

    if anomalias:
        print("Anomalias encontradas:")
        for anomalia in anomalias:
            print(f"- {anomalia}")
    else:
        print("Nenhuma anomalia encontrada.")

## 3. Dados simulados

Foram criados dois cenários para demonstrar as duas respostas possíveis do algoritmo.

In [2]:
cenario_seguro = {
    "temperatura_interna": 22,
    "temperatura_externa": 28,
    "integridade_estrutural": 1,
    "energia": 92,
    "pressao_tanques": {"combustível": 300, "oxidante": 305},
    "modulos_criticos": {
        "propulsão": True,
        "navegação": True,
        "comunicação": True,
        "suporte de vida": True,
    },
}

cenario_com_anomalias = {
    "temperatura_interna": 31,
    "temperatura_externa": 28,
    "integridade_estrutural": 1,
    "energia": 54,
    "pressao_tanques": {"combustível": 250, "oxidante": 300},
    "modulos_criticos": {
        "propulsão": False,
        "navegação": True,
        "comunicação": True,
        "suporte de vida": True,
    },
}

## 4. Execução das verificações

In [3]:
mostrar_resultado("CENÁRIO SEGURO", cenario_seguro)
mostrar_resultado("CENÁRIO COM ANOMALIAS", cenario_com_anomalias)


=== CENÁRIO SEGURO ===
Resultado: PRONTO PARA DECOLAR
Nenhuma anomalia encontrada.

=== CENÁRIO COM ANOMALIAS ===
Resultado: DECOLAGEM ABORTADA
Anomalias encontradas:
- Temperatura interna fora da faixa: 31 °C
- Energia abaixo do mínimo: 54%
- Pressão do tanque combustível fora da faixa: 250 kPa
- Módulo crítico com falha: propulsão


## 5. Análise energética

A análise usa capacidade total de 120 kWh, carga atual de 92%, consumo de 30 kWh na decolagem e perdas de 8%.

In [4]:
energia_inicial, perdas, energia_restante, autonomia = calcular_energia(
    capacidade_kwh=120,
    carga_atual=92,
    consumo_decolagem_kwh=30,
    perdas_percentuais=8,
)

print(f"Energia inicial: {energia_inicial:.2f} kWh")
print(f"Perdas estimadas: {perdas:.2f} kWh")
print(f"Energia restante após a decolagem: {energia_restante:.2f} kWh")
print(f"Autonomia restante: {autonomia:.2f}% da capacidade total")

Energia inicial: 110.40 kWh
Perdas estimadas: 8.83 kWh
Energia restante após a decolagem: 71.57 kWh
Autonomia restante: 59.64% da capacidade total


## 6. Análise assistida por IA

### Prompt enviado à IA

> Analise os dois cenários de telemetria apresentados.  
> **Cenário seguro:** temperatura interna 22 °C, temperatura externa 28 °C, integridade estrutural 1, energia 92%, pressões de 300 kPa e 305 kPa e todos os módulos críticos funcionando.  
> **Cenário com anomalias:** temperatura interna 31 °C, temperatura externa 28 °C, integridade estrutural 1, energia 54%, pressões de 250 kPa e 300 kPa e módulo de propulsão com falha.  
> Classifique os dados, identifique possíveis anomalias e sugira os riscos e as ações necessárias antes da decolagem.

### Resposta da IA

A análise da telemetria classifica o **Cenário Seguro** como nominal para a decolagem, com status *GO* (**PRONTO PARA DECOLAR**). O **Cenário com Anomalias** apresenta condições críticas que exigem o aborto da missão, com status *NO-GO* (**DECOLAGEM ABORTADA**).

A classificação usa exclusivamente as faixas de segurança definidas nesta simulação acadêmica.

#### Classificação e identificação de anomalias

| Parâmetro | Cenário seguro | Cenário com anomalias | Classificação da IA |
| --- | --- | --- | --- |
| **Temperatura interna** | 22 °C | 31 °C | **Crítico:** acima do limite máximo de 27 °C. |
| **Temperatura externa** | 28 °C | 28 °C | **Nominal:** dentro da faixa de -10 °C a 45 °C. |
| **Integridade estrutural** | 1 (íntegra) | 1 (íntegra) | **Nominal:** o indicador binário confirma a integridade estrutural. |
| **Energia** | 92% | 54% | **Crítico:** abaixo do mínimo de 80%. |
| **Pressões dos tanques** | 300 kPa / 305 kPa | 250 kPa / 300 kPa | **Crítico:** o tanque de combustível está abaixo do mínimo de 280 kPa; o oxidante permanece nominal. |
| **Módulos críticos** | Todos operantes | Propulsão com falha | **Crítico:** um módulo obrigatório está inoperante. |

#### Análise de riscos do cenário com anomalias

Os dados disponíveis não permitem afirmar a causa exata das falhas, mas permitem apontar riscos e verificações necessárias:

- **Controle térmico:** a temperatura interna de 31 °C excede o limite e pode indicar geração excessiva de calor ou resfriamento insuficiente.
- **Energia:** o nível de 54% reduz a margem disponível para propulsão, navegação, comunicação e suporte de vida.
- **Pressurização do combustível:** a leitura de 250 kPa pode estar relacionada a vazamento, sensor ou sistema de pressurização e deve ser confirmada por inspeção.
- **Propulsão:** a falha registrada impede uma decolagem segura, pois o módulo é indispensável para produzir e controlar o empuxo.

#### Ações necessárias antes da decolagem

1. Suspender imediatamente a decolagem.
2. Inspecionar o sistema de arrefecimento e identificar a origem da temperatura elevada.
3. Verificar baterias, conexões e cargas elétricas e restabelecer pelo menos 80% de energia, preferencialmente com margem adicional de segurança.
4. Testar a linha, o sensor e o sistema de pressurização do tanque de combustível.
5. Executar o diagnóstico completo do módulo de propulsão e corrigir a falha encontrada.
6. Repetir todas as leituras e autorizar uma nova tentativa somente quando nenhuma anomalia permanecer.

**Observação:** a inteligência artificial foi utilizada como ferramenta de apoio. A decisão final deve permanecer sob responsabilidade de profissionais qualificados e dos procedimentos oficiais de segurança.


## 7. Conclusão

O algoritmo encontrou corretamente as anomalias do segundo cenário e impediu a decolagem. O primeiro cenário foi aprovado porque todos os valores estavam dentro das faixas seguras.